In [ ]:
import os
import pandas as pd

HF_PATH = "hf://datasets/qiaojin/PubMedQA/pqa_labeled/train-00000-of-00001.parquet"
LOCAL_PATH = "pqa_labeled.parquet"

if os.path.exists(LOCAL_PATH):
    df = pd.read_parquet(LOCAL_PATH)
    print("Loaded local copy:", LOCAL_PATH)
else:
    # first run only
    df = pd.read_parquet(HF_PATH)
    df.to_parquet(LOCAL_PATH, index=False)
    print("Downloaded and saved:", LOCAL_PATH)

selected_df = df[[
    "pubid",
    "question",
    "context",
    "long_answer",
    "final_decision"
]].copy()

selected_df = selected_df.rename(columns={"pubid": "pub_id"})

In [ ]:
import re
 
CORE_MODALS = {"can", "could", "may", "might", "will", "would", "shall", "should", "must"}
DO_BE_HAVE = {"do", "does", "did",
              "am", "is", "are", "was", "were",
              "have", "has", "had"}
 
AUX = CORE_MODALS | DO_BE_HAVE
 
def is_well_formed(q, aux=AUX):
    q = " ".join(str(q).split())
 
    # a question must end with a question mark
    if not q.endswith("?"):
        return False
        
    # the question itself is not changed; for the check, look only at the clause after the last ": ", "--" or " - " (hyphens inside words are not split)
    last = re.split(r":\s|--|\s-\s", q)[-1].strip()
 
    # split that clause into words
    words = last.split()
 
    # if the clause has no words (empty piece), it can't be a question, so reject it
    if not words:
        return False
 
    # keep letters only in the first word (drops brackets, quotes, punctuation and digits)
    raw = re.sub(r"[^A-Za-z]", "", words[0])
 
    # reject the abbreviation "AM" (morning). otherwise it would match the verb "am" after lowercasing
    if raw == "AM":
        return False
 
    # the first word of the clause must be an allowed auxiliary or modal verb (compared in lowercase)
    return raw.lower() in aux
 
# questions ending in "A or B?" give a choice, so yes/no/maybe is not a valid answer
def is_alternative_question(q):
    return bool(re.search(r"\bor\s+[\w'’\-]+\s*\?$", q.strip()))
 
# the context contains a section labelled as a conclusion (which would give the answer away)
def has_conclusion_label(ctx):
    try:
        return any("CONCLUSION" in str(label).upper() for label in ctx["labels"])
    except (TypeError, KeyError):
        return False
 
# copy so selected_df stays untouched
clean_df = selected_df.copy()
 
# convert double spaces, tabs, line breaks to single spaces
clean_df["question"] = clean_df["question"].map(lambda q: " ".join(str(q).split()) if pd.notna(q) else q)
 
# one label per question
# if a question breaks a rule, the reason is stated
reasons = pd.Series("", index=clean_df.index)
 
rules = [
    # 1. question or label is empty
    ("missing question or label", clean_df[["question", "final_decision"]].isna().any(axis=1)),
    # 2. label is anything other than yes / no / maybe
    ("label not yes/no/maybe", ~clean_df["final_decision"].isin(["yes", "no", "maybe"])),
    # 3. remove duplicates
    ("duplicate question", clean_df["question"].duplicated(keep="first")),
    # 4. is_well_formed() returns false
    ("not phrased as a yes/no question", ~clean_df["question"].map(lambda q: isinstance(q, str) and is_well_formed(q))),
    # 5. ends in "A or B?" (a choice, not a yes/no question)
    ("alternative (either/or) question", clean_df["question"].map(lambda q: isinstance(q, str) and is_alternative_question(q))),
    # 6. the context has a section labelled as a conclusion
    ("conclusion in context labels", clean_df["context"].map(has_conclusion_label)),
]
 
# apply the rules in order
for name, mask in rules:
    reasons[mask & (reasons == "")] = name
clean_df["exclusion_reason"] = reasons
 
# pool refers to the questions that can be selected / don't break rules
pool = clean_df[clean_df["exclusion_reason"] == ""]
excluded = clean_df[clean_df["exclusion_reason"] != ""]
 
# display # of qs kept, removed, and why
print(clean_df["exclusion_reason"].replace("", "kept").value_counts(), "\n")
 
# sanity check
print("class counts before:", selected_df["final_decision"].value_counts().to_dict())
print("class counts after: ", pool["final_decision"].value_counts().to_dict(), "\n")
 
# print excluded questions
for _, r in excluded.iterrows():
    print(f"[{r['pub_id']}] ({r['final_decision']}) {r['exclusion_reason']}: {r['question']}")
 
OUTPUT_DIR = "question_samples_60"
os.makedirs(OUTPUT_DIR, exist_ok=True)
excluded.to_csv(os.path.join(OUTPUT_DIR, "excluded_questions.csv"), index=False)
print("Saved", os.path.join(OUTPUT_DIR, "excluded_questions.csv"))

In [ ]:
import json
import os
import datetime
 
confidence_prompt = "State your confidence score from 0 to 100, indicating how confident you are that the answer is correct. 0 indicates no confidence, and 100 indicates full confidence. Please ensure explanations are between 2 and 3 sentences. Answer based only on the evidence provided in the context above, without relying on outside knowledge. Provide your response in the format:\n\nAnswer: Yes / No / Maybe\nExplanation: [explanation]\nConfidence: X/100" 

emotion_templates = { 
    "N1": "I would like to learn more about this topic: ",
    "N2": "Please provide an answer to the following question: ",
    "N3": "I would like to know more about the following question: ",
    "F1": "I am nervous about this question and would like an answer: ",
    "F2": "I am feeling anxious and would like more information about this: ",
    "F3": "I am concerned about this situation and would like to know more: ",
    "J1": "I am happy and would like to learn more about this topic: ",
    "J2": "I am excited to learn the answer to the following question: ",
    "J3": "I am feeling hopeful about this and would like to know more: "
}
 
# sampling settings
SEED = 42
N_QUESTIONS = 60
ALLOCATION = {"yes": 20, "no": 20, "maybe": 20} # equal allocation, n = 60
SAVE_SAMPLE = True

# output files: one shared template (both models' response columns), one info file, one exclusion log
OUTPUT_DIR = "question_samples_60"
SAMPLE_PATH = os.path.join(OUTPUT_DIR, "final_sample_60_prompts.csv")
INFO_PATH = os.path.join(OUTPUT_DIR, "final_sample_60_prompts_info.json")
LOG_PATH = os.path.join(OUTPUT_DIR, "final_sample_60_excluded.csv")
 
 
# abstract text from the PubMedQA "context" field: only "contexts" with its section labels
# (never long_answer, final_decision, meshes or the *_pred fields), and any Conclusion section removed
def is_conclusion(label):
    return "CONCLUSION" in str(label).upper()
 
def format_context(ctx):
    return "\n\n".join(f"{str(label).title()}: {text}"
                        for label, text in zip(ctx["labels"], ctx["contexts"]) if not is_conclusion(label))
 
 
assert sum(ALLOCATION.values()) == N_QUESTIONS, f"Allocation must add up to {N_QUESTIONS}"
assert "context" in pool.columns, "pool has no 'context' column: add \"context\" to selected_df in the loading cell"
available = pool["final_decision"].value_counts().to_dict()
for lab, k in ALLOCATION.items():
    assert available.get(lab, 0) >= k, f"Only {available.get(lab, 0)} '{lab}' questions in the cleaned pool, need {k}"
 
parts = [pool[pool["final_decision"] == lab].sample(n=k, random_state=SEED) for lab, k in ALLOCATION.items()]
sample_60 = (pd.concat(parts)
             .sample(frac=1, random_state=SEED) # shuffle so ids are not grouped by class
             .drop(columns="exclusion_reason")
             .reset_index(drop=True))
 
# add id column
sample_60.insert(0, "id", range(1, len(sample_60) + 1))
 
# abstract text; any section labelled as a conclusion is removed (and counted, for the methods section)
sample_60["context_text"] = sample_60["context"].map(format_context)
n_conclusion_sections = int(sample_60["context"].map(lambda c: sum(is_conclusion(l) for l in c["labels"])).sum())
print(f"Conclusion sections removed from the contexts: {n_conclusion_sections}")
assert not sample_60["context_text"].str.contains(r"^Conclusions?:", regex=True, case=False).any()
assert not sample_60["context_text"].str.contains(r"\n\nConclusions?:", regex=True, case=False).any()
contexts = sample_60.set_index("id")["context_text"].to_dict()
sample_60 = sample_60.drop(columns=["context", "context_text"]) # same columns as the original question-only template
 
sample_60.insert(5, "emotional_question", "")
sample_60.insert(1, "key", "")
for t in range(1, 12):
    sample_60[f"claude_response_t{t}"] = ""
for t in range(1, 12):
    sample_60[f"gemini_response_t{t}"] = ""
 
# add emotional context, the abstract and the confidence prompt
new_rows = []
for i, row in sample_60.iterrows():
    for key, prompt in emotion_templates.items():
        new_row = row.to_dict()
        new_row["key"] = str(row["id"]) + key
        new_row["context"] = contexts[row["id"]]
        # order: Context -> emotional sentence -> question -> answer instructions
        new_row["emotional_question"] = ("Context:\n" + contexts[row["id"]] + "\n\n"
                                         + prompt + "\n" + row["question"] + "\n\n" + confidence_prompt)
        new_rows.append(new_row)
 
sample_60 = pd.DataFrame(new_rows)
 
# checks: every prompt has the abstract and the question, and never the answer
for r in sample_60.itertuples():
    assert r.emotional_question.startswith("Context:\n") and r.question in r.emotional_question, f"{r.key}: prompt is incomplete"
    assert r.emotional_question.index(r.question) > r.emotional_question.index(emotion_templates[r.key[-2:]]), f"{r.key}: wrong order"
    assert str(r.long_answer) not in r.emotional_question, f"{r.key}: the conclusion is in the prompt"
 
pd.set_option("display.max_colwidth", None)
 
# one row per question (needed by the save block and the printout below)
chosen = sample_60.drop_duplicates("id")
 
# print selected questions
print(f"Seed {SEED} | {len(chosen)} questions | pool class counts: {available}")
print("Sample class counts:", chosen["final_decision"].value_counts().to_dict(), "\n")
for _, r in chosen.iterrows():
    print(f"[{r['id']}] ({r['final_decision']}) {r['question']}\n")
words = pd.Series(contexts).str.split().str.len()
print(f"Abstract length: {words.mean():.0f} words on average (min {words.min()}, max {words.max()})\n")
print("----- Example prompt (1N1) -----")
print(sample_60.loc[sample_60["key"] == "1N1", "emotional_question"].iloc[0], "\n")
 
# save both files (only when SAVE_SAMPLE is True, and never over an existing file)
base_info = {
    "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "seed": SEED,
    "allocation": ALLOCATION,
    "n_questions": N_QUESTIONS,
    "pool_class_counts": {k: int(v) for k, v in available.items()},
    "sample_class_counts": {k: int(v) for k, v in chosen["final_decision"].value_counts().items()},
    "cleaning_counts": {k: int(v) for k, v in clean_df["exclusion_reason"].replace("", "kept").value_counts().items()},
    "question_ids": [int(i) for i in chosen["id"]],
    "pub_ids": [int(i) for i in chosen["pub_id"]],
}

setting = ("context: PubMedQA 'contexts' field with its section labels; order = Context -> emotional sentence -> "
           "question -> answer instructions; long_answer, final_decision, meshes, *_pred fields and any Conclusion "
           "section excluded")

# excluded questions from the cleaning cell, with their reason
reasons = clean_df["exclusion_reason"].fillna("").astype(str)
excluded = clean_df[reasons != ""]
log_cols = [c for c in ["pub_id", "question", "final_decision", "exclusion_reason"]
            if c in excluded.columns]
excluded_log = excluded[log_cols].sort_values(["exclusion_reason", "pub_id"])
print(f"Excluded during cleaning: {len(excluded_log)} questions")
print(excluded_log["exclusion_reason"].value_counts().to_string(), "\n")

if not SAVE_SAMPLE:
    print("Not saved (SAVE_SAMPLE is False). Read the questions, then set it to True and run again.")
else:
    if os.path.exists(SAMPLE_PATH):
        # never overwrite a saved template, but check it is the same sample
        saved = pd.read_csv(SAMPLE_PATH).drop_duplicates("id")
        same = saved["pub_id"].astype(int).tolist() == base_info["pub_ids"]
        print(f"{SAMPLE_PATH} already exists - nothing overwritten. Same questions as this sample: {same}")
        assert same, f"{SAMPLE_PATH} has different questions from this sample - check the seed and the cleaning cell"
    else:
        info = {**base_info, "setting": setting, "models": ["claude", "gemini"],
                "conclusion_sections_removed": n_conclusion_sections,
                "n_excluded": len(excluded)}
        # info first, so a crash never leaves a CSV without it
        with open(INFO_PATH, "w", encoding="utf-8") as f:
            json.dump(info, f, indent=2)
        sample_60.to_csv(SAMPLE_PATH, index=False)
        chosen[["id", "pub_id", "question", "final_decision"]].to_csv(os.path.join(OUTPUT_DIR, "final_sample_60_questions.csv"), index=False)
        print("Saved", SAMPLE_PATH, "and", INFO_PATH)
    print("Sample class counts:", base_info["sample_class_counts"])